# 05 — RQ3: The Solar Duck Curve

**Question:** What is the solar "duck curve" signature in Austria, and how has it
deepened as solar scaled (2019→2024)?

The "duck curve" is the daily shape of **net load** — electricity demand minus solar
generation — the load that everything *other* than solar (hydro, gas, imports) has to
serve. As solar floods in around midday, net load sinks into a **belly**; as the sun
sets into the evening demand peak, solar vanishes and net load climbs a steep **evening
ramp**. The silhouette — belly, then neck — looks like a duck.

This is the operational flip-side of RQ1. There, solar scaling ~5× was a line on an
annual chart; here, that same growth shows up as a *within-day* problem — a deeper midday
trough and a sharper evening ramp that dispatchable generation must follow. It also sets
up RQ4: the belly hours are exactly when day-ahead prices crater (the merit-order effect).

## Data

Two hourly ENTSO-E tables, both stored UTC, joined 1:1 on `ts_utc`:

| Table | Grain | Source | Column used |
|---|---|---|---|
| `demand`     | hourly | ENTSO-E (Austrian Power Grid / APG), resampled from 15-min | `demand_mw` |
| `generation` | hourly | ENTSO-E (APG), resampled from 15-min | `mw` where `fuel_type = 'Solar'`, `flow = 'generation'` |

Solar sits in the long-format `generation` table (one row per fuel × flow), so it's
filtered to its own series before the join. Both series are **national** totals — unlike
RQ2 (national demand vs Vienna-only temperature), there's no single-station proxy here.
The `demand ⋈ solar` join is exactly one-to-one (52,608 hours, no orphan hour — verified
in Cell B). All clock-dependent aggregation converts to `Europe/Vienna` at the display
layer.

## Method — diurnal profiles + year-over-year

**Grain is HOURLY — deliberately the opposite of RQ2.** RQ2 aggregated to daily means
*to remove* the hour-of-day cycle. Here that cycle *is* the phenomenon: the duck is a
within-day shape, so we stay hourly and look at the average profile across the 24
Vienna-local hours.

- **Diurnal profiles.** Average net load by hour-of-day isolates the characteristic
  belly + ramp shape. Splitting by season confirms the duck is a *summer* phenomenon —
  winter solar barely dents net load.
- **Year-over-year.** Overlaying each year's summer profile shows the belly deepening and
  the ramp steepening as solar capacity grew — the "how has it grown" half of the question.

**Solar-only (headline), with a wind footnote.** The duck is canonically a *solar*
signature: solar is clock-locked (zero at night, midday spike), so it carves a
fixed-shape belly. Wind is weather-driven with a near-flat diurnal profile, so netting it
out shifts net load *down* without changing its *shape*. We net solar only for the
headline duck; total variable-renewable net load (demand − solar − wind) appears as
operational context, not as the duck itself.

| Step | What | Output |
|---|---|---|
| 1 | Build net load, eyeball the diurnal shape (summer vs winter, avg vs 2024) | Diurnal profiles |
| 2 | Quantify the signature (belly depth, evening ramp rate, timing) + show its growth | Year-over-year profiles + metrics |
| 3 | Headline visual + 2-sentence finding | Duck-curve growth figure |

**Note — what an averaged profile hides.** Hour-of-day means smooth away day-to-day
weather, so the headline shape is a *typical* day, not any real one; the deepest
individual bellies are sharper. And the 2019→2024 belly deepening conflates two things —
more solar *and* the post-2022 demand decline (RQ2) — separated where it matters in Step 2.

In [ ]:
# Cell A: connect
import duckdb
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
DB_PATH = PROJECT_ROOT / "data" / "processed" / "austria_energy.duckdb"
con = duckdb.connect(str(DB_PATH))

In [ ]:
# Cell B: solar series + join integrity + window
# A Common Table Expression (CTE) isolates the Solar generation series; we then
# join it 1:1 to demand on ts_utc. Matching counts ⇒ no demand hour was silently
# dropped because solar happened to be unreported that hour.
con.sql("""
WITH solar AS (
    SELECT ts_utc, mw AS solar_mw
    FROM generation
    WHERE fuel_type = 'Solar' AND flow = 'generation'
)
SELECT 'demand'            AS tbl, COUNT(*) AS n, MIN(ts_utc) AS lo, MAX(ts_utc) AS hi
FROM demand
UNION ALL
SELECT 'solar',                  COUNT(*),       MIN(ts_utc),       MAX(ts_utc)
FROM solar
UNION ALL
SELECT 'demand JOIN solar',      COUNT(*),       MIN(ts_utc),       MAX(ts_utc)
FROM demand d JOIN solar s USING (ts_utc)
""").df()

In [ ]:
# Cell C: average diurnal shape — summer vs winter
# Net load = demand − solar, hourly, Vienna-LOCAL. EXTRACT runs on the local
# wall-clock (the AT TIME ZONE result), sidestepping the session-tz gotcha.
import sys
from typing import cast
import matplotlib.pyplot as plt

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.viz import set_house_style, PALETTE
set_house_style()

diurnal = con.sql("""
WITH solar AS (
    SELECT ts_utc, mw AS solar_mw
    FROM generation
    WHERE fuel_type = 'Solar' AND flow = 'generation'
),
net_load AS (
    SELECT
        d.ts_utc AT TIME ZONE 'Europe/Vienna'  AS ts_local,   -- convert once
        d.demand_mw,
        d.demand_mw - s.solar_mw               AS net_load_mw
    FROM demand d
    JOIN solar s USING (ts_utc)               -- 1:1 (verified in Cell B)
)
SELECT
    EXTRACT(hour FROM ts_local)                                                AS hour_local,
    AVG(demand_mw)   FILTER (WHERE EXTRACT(month FROM ts_local) IN (6, 7, 8))   AS demand_summer,
    AVG(net_load_mw) FILTER (WHERE EXTRACT(month FROM ts_local) IN (6, 7, 8))   AS net_summer,
    AVG(demand_mw)   FILTER (WHERE EXTRACT(month FROM ts_local) IN (12, 1, 2))  AS demand_winter,
    AVG(net_load_mw) FILTER (WHERE EXTRACT(month FROM ts_local) IN (12, 1, 2))  AS net_winter
FROM net_load
GROUP BY hour_local
ORDER BY hour_local
""").df()

h = diurnal["hour_local"]
fig, (ax_s, ax_w) = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

ax_s.fill_between(h, diurnal["net_summer"], diurnal["demand_summer"],
                  color=PALETTE["solar"], alpha=0.5, label="Solar (carved out)")
ax_s.plot(h, diurnal["demand_summer"], color=PALETTE["demand"], lw=1.8, label="Demand")
ax_s.plot(h, diurnal["net_summer"], color="#1A1A1A", lw=2.2, label="Net load (demand − solar)")
ax_s.set(title="Summer (Jun–Aug)", xlabel="hour of day (local)", ylabel="MW")

ax_w.fill_between(h, diurnal["net_winter"], diurnal["demand_winter"],
                  color=PALETTE["solar"], alpha=0.5)
ax_w.plot(h, diurnal["demand_winter"], color=PALETTE["demand"], lw=1.8)
ax_w.plot(h, diurnal["net_winter"], color="#1A1A1A", lw=2.2)
ax_w.set(title="Winter (Dec–Feb)", xlabel="hour of day (local)")

ax_s.legend(loc="lower center", fontsize=9)
fig.suptitle("Austria — average net-load diurnal shape: the duck curve is a summer phenomenon", y=1.02)
plt.tight_layout()
plt.show()

# eyeball aid (formal metrics come in step 2)
# belly = the MIDDAY depression (10–16h), NOT the 4am demand trough, which is
# the global minimum of net load but unrelated to solar.
midday = diurnal[diurnal["hour_local"].between(10, 16)]
evening = diurnal[diurnal["hour_local"].between(17, 22)]
belly_mw  = midday["net_summer"].min()
belly_h   = cast(int, midday.loc[midday["net_summer"].idxmin(), "hour_local"])
peak_mw   = evening["net_summer"].max()
peak_h    = cast(int, evening.loc[evening["net_summer"].idxmax(), "hour_local"])
print(f"summer midday belly:  {belly_mw:,.0f} MW at {belly_h:02d}:00")
print(f"summer evening peak:  {peak_mw:,.0f} MW at {peak_h:02d}:00")
print(f"belly → peak rise:    {peak_mw - belly_mw:,.0f} MW over {peak_h - belly_h} h")
display(diurnal.round(0))

In [ ]:
# Cell D — net load by season: 2019–2024 avg vs 2024, one panel per season
# Generalises Cells D (summer) and E (winter) into one figure. CASE tags the four
# meteorological seasons in a single pass; FILTER (WHERE yr = 2024) pulls the 2024
# line. sharey=True ⇒ belly depths are comparable across panels (deepest = summer).
diurnal_seasons = con.sql("""
WITH solar AS (
    SELECT ts_utc, mw AS solar_mw
    FROM generation
    WHERE fuel_type = 'Solar' AND flow = 'generation'
),
net_load AS (
    SELECT
        d.ts_utc AT TIME ZONE 'Europe/Vienna'  AS ts_local,
        d.demand_mw - s.solar_mw               AS net_load_mw
    FROM demand d
    JOIN solar s USING (ts_utc)
),
tagged AS (
    SELECT
        EXTRACT(hour FROM ts_local)  AS hour_local,
        EXTRACT(year FROM ts_local)  AS yr,
        CASE                                            -- meteorological seasons
            WHEN EXTRACT(month FROM ts_local) IN (12, 1, 2)  THEN 'Winter'
            WHEN EXTRACT(month FROM ts_local) IN (3, 4, 5)   THEN 'Spring'
            WHEN EXTRACT(month FROM ts_local) IN (6, 7, 8)   THEN 'Summer'
            WHEN EXTRACT(month FROM ts_local) IN (9, 10, 11) THEN 'Autumn'
        END                          AS season,
        net_load_mw
    FROM net_load
)
SELECT
    season,
    hour_local,
    AVG(net_load_mw)                            AS net_all,
    AVG(net_load_mw) FILTER (WHERE yr = 2024)   AS net_2024
FROM tagged
GROUP BY season, hour_local
ORDER BY season, hour_local
""").df()

seasons = ["Spring", "Summer", "Autumn", "Winter"]   # calendar order across the 2×2

fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=True, sharey=True)
for ax, season in zip(axes.flat, seasons):
    sub = diurnal_seasons[diurnal_seasons["season"] == season].sort_values("hour_local")
    ax.plot(sub["hour_local"], sub["net_all"], color=PALETTE["muted"], lw=2,
            marker="o", ms=3.5, label="2019–2024 avg")
    ax.plot(sub["hour_local"], sub["net_2024"], color=PALETTE["accent"], lw=2.2,
            marker="o", ms=3.5, label="2024")
    ax.set_title(season)

for ax in axes[-1, :]:
    ax.set_xlabel("hour of day (local)")
for ax in axes[:, 0]:
    ax.set_ylabel("avg net load (MW)")
axes[0, 0].legend(loc="upper right", fontsize=9)

fig.suptitle("Net load by season — the duck is in every season, deepest in summer", y=1.02)
plt.tight_layout()
plt.show()

display(diurnal_seasons.round(0))

In [ ]:
# Cell E — justify solar-only: diurnal SHAPE of solar vs wind
# The duck is a SOLAR signature because solar is clock-locked. Show it empirically,
# not by assertion: compare each source's hour-of-day SHAPE, normalised to its own
# daily mean so both sit on one scale. Solar swings 0 → sharp midday peak; wind ~flat.
# NB ENTSO-E labels Austrian wind 'Wind Onshore' (no offshore). If wind_mw comes back
# empty, check: SELECT DISTINCT fuel_type FROM generation WHERE flow = 'generation'.
diurnal_vre = con.sql("""
WITH vre AS (
    SELECT
        ts_utc AT TIME ZONE 'Europe/Vienna'  AS ts_local,
        fuel_type,
        mw
    FROM generation
    WHERE flow = 'generation' AND fuel_type IN ('Solar', 'Wind Onshore')
)
SELECT
    EXTRACT(hour FROM ts_local)                        AS hour_local,
    AVG(mw) FILTER (WHERE fuel_type = 'Solar')         AS solar_mw,
    AVG(mw) FILTER (WHERE fuel_type = 'Wind Onshore')  AS wind_mw
FROM vre
GROUP BY hour_local
ORDER BY hour_local
""").df()

# normalise each to its own 24-hour mean → compares SHAPE, not magnitude
diurnal_vre["solar_norm"] = diurnal_vre["solar_mw"] / diurnal_vre["solar_mw"].mean()
diurnal_vre["wind_norm"]  = diurnal_vre["wind_mw"]  / diurnal_vre["wind_mw"].mean()

fig, ax = plt.subplots()
ax.plot(diurnal_vre["hour_local"], diurnal_vre["solar_norm"], color=PALETTE["solar"],
        lw=2.2, marker="o", ms=4, label="Solar")
ax.plot(diurnal_vre["hour_local"], diurnal_vre["wind_norm"], color=PALETTE["wind"],
        lw=2.2, marker="o", ms=4, label="Wind")
ax.axhline(1.0, color=PALETTE["muted"], lw=0.8, ls="--")   # each series' own daily mean
ax.set(title="Why solar-only: solar is clock-locked, wind is flat",
       xlabel="hour of day (local)", ylabel="generation ÷ own daily mean")
ax.legend()
plt.tight_layout()
plt.show()

# one-line evidence: how far each swings from its own daily mean
for name, col in [("solar", "solar_norm"), ("wind", "wind_norm")]:
    lo, hi = diurnal_vre[col].min(), diurnal_vre[col].max()
    print(f"{name:5s}: ranges {lo:.2f}× – {hi:.2f}× its daily mean")

solar is clock-locked, wind is flat, so netting wind would shift net-load level, not shape — we net solar only.

In [ ]:
# Cell F — quantify the duck shape of one diurnal net-load profile
def duck_metrics(profile: pd.DataFrame) -> pd.Series:
    """
    Duck-curve metrics for one diurnal net-load profile (one season × year).

    Parameters
    ----------
    profile : pd.DataFrame
        24 rows, columns 'hour_local' (0–23) and 'net_load_mw' (hourly mean).

    Returns
    -------
    pd.Series:
        belly_depth_mw  morning shoulder − midday trough. The solar-driven dent.
                        Level-invariant (the demand decline cancels), so its
                        *rise over years* is the solar signal — read the slope.
        trough_mw       absolute midday-trough level (informative, but partly
                        falls with the demand decline → don't read as solar).
        ramp_rate_mwh   avg evening ramp = (evening peak − trough) / Δhours.
        max_ramp_mwh    steepest single-hour climb between trough and peak.
        trough_hour     hour of the midday trough.
        peak_hour       hour of the evening peak.
    """
    if len(profile) < 24:          # partial profile (boundary-hour leak) → not a real day
            return pd.Series({k: float("nan") for k in
                ["belly_depth_mw", "trough_mw", "ramp_rate_mwh",
                 "max_ramp_mwh", "trough_hour", "peak_hour"]})

    p = profile.set_index("hour_local")["net_load_mw"].sort_index()

    morning = p.loc[6:10]      # morning shoulder window (Vienna-local hours)
    midday  = p.loc[11:15]     # belly window
    evening = p.loc[17:22]     # evening peak window

    shoulder_mw = morning.max()
    trough_mw   = midday.min()
    trough_hour = int(midday.idxmin())
    peak_mw     = evening.max()
    peak_hour   = int(evening.idxmax())

    return pd.Series({
        "belly_depth_mw": round(shoulder_mw - trough_mw),
        "trough_mw":      round(trough_mw),
        "ramp_rate_mwh":  round((peak_mw - trough_mw) / (peak_hour - trough_hour)),
        "max_ramp_mwh":   round(p.loc[trough_hour:peak_hour].diff().max()),
        "trough_hour":    trough_hour,
        "peak_hour":      peak_hour,
    })

In [ ]:
# Cell G — per-(season, year) profiles → metric table
profiles = con.sql("""
WITH solar AS (
    SELECT ts_utc, mw AS solar_mw
    FROM generation
    WHERE fuel_type = 'Solar' AND flow = 'generation'
),
net_load AS (
    SELECT
        d.ts_utc AT TIME ZONE 'Europe/Vienna'  AS ts_local,
        d.demand_mw - s.solar_mw               AS net_load_mw
    FROM demand d
    JOIN solar s USING (ts_utc)
),
tagged AS (
    SELECT
        EXTRACT(hour  FROM ts_local) AS hour_local,
        EXTRACT(year  FROM ts_local) AS yr,
        CASE
            WHEN EXTRACT(month FROM ts_local) IN (12, 1, 2)  THEN 'Winter'
            WHEN EXTRACT(month FROM ts_local) IN (3, 4, 5)   THEN 'Spring'
            WHEN EXTRACT(month FROM ts_local) IN (6, 7, 8)   THEN 'Summer'
            WHEN EXTRACT(month FROM ts_local) IN (9, 10, 11) THEN 'Autumn'
        END                          AS season,
        net_load_mw
    FROM net_load
    WHERE ts_local < TIMESTAMP '2025-01-01'   -- drop the 1-h UTC→local spill into 2025
)
SELECT season, yr AS year, hour_local, AVG(net_load_mw) AS net_load_mw
FROM tagged
GROUP BY season, yr, hour_local
ORDER BY season, yr, hour_local
""").df()

rows = []
for (season, year), grp in profiles.groupby(["season", "year"]):
    m = duck_metrics(grp)
    rows.append(pd.Series({"season": season, "year": int(year), **m.to_dict()}))
metrics = pd.DataFrame(rows)

season_order = ["Spring", "Summer", "Autumn", "Winter"]   # calendar order
metrics["season"] = pd.Categorical(metrics["season"], season_order, ordered=True)
metrics = metrics.sort_values(["season", "year"]).reset_index(drop=True)

display(metrics)

In [ ]:
# Cell I — belly depth by year, one line per season
season_colors = {"Spring": PALETTE["wind"], "Summer": PALETTE["temp"],
                 "Autumn": PALETTE["solar"], "Winter": PALETTE["hydro"]}

fig, ax = plt.subplots()
for season in season_order:
    sub = metrics[metrics["season"] == season]
    ax.plot(sub["year"], sub["belly_depth_mw"], marker="o", lw=2,
            color=season_colors[season], label=season)
ax.set(title="Duck belly depth by year — every season deepening, summer steepest",
       xlabel="year", ylabel="belly depth (MW): morning shoulder − midday trough")
ax.legend(title="season")
plt.tight_layout()
plt.show()

In [ ]:
# Cell J — headline: the summer duck deepening, 2019→2024
import numpy as np
from src.viz import save_headline_fig

summer = profiles[profiles["season"] == "Summer"]
years  = sorted(summer["year"].unique())
colors = plt.cm.YlOrRd(np.linspace(0.35, 0.95, len(years)))   # light → deep red

fig, ax = plt.subplots(figsize=(11, 6))
for yr, c in zip(years, colors):
    sub = summer[summer["year"] == yr].sort_values("hour_local")
    ax.plot(sub["hour_local"], sub["net_load_mw"], color=c,
            lw=3.0 if yr == 2024 else 1.8, marker="o", ms=3, label=str(yr))

ax.set(title="Austria — the summer duck curve deepens, 2019→2024",
       xlabel="hour of day (local)", ylabel="avg summer net load (MW)")
ax.legend(title="year", ncol=2, fontsize=9)

# headline numbers, read from the metric table
m19 = metrics[(metrics["season"] == "Summer") & (metrics["year"] == 2019)].iloc[0]
m24 = metrics[(metrics["season"] == "Summer") & (metrics["year"] == 2024)].iloc[0]
txt = (f"belly depth    {m19.belly_depth_mw:.0f} → {m24.belly_depth_mw:.0f} MW\n"
       f"evening ramp   {m19.ramp_rate_mwh:.0f} → {m24.ramp_rate_mwh:.0f} MW/h\n"
       f"steepest hour  {m19.max_ramp_mwh:.0f} → {m24.max_ramp_mwh:.0f} MW/h")
ax.text(0.02, 0.04, txt, transform=ax.transAxes, ha="left", va="bottom",
        fontsize=8.5, family="monospace",
        bbox=dict(boxstyle="round,pad=0.5", fc="white", ec=PALETTE["muted"], lw=0.8))

plt.tight_layout()
save_headline_fig(fig, "rq3_duck_curve")
plt.show()

In [ ]:
# Close the connection
con.close()

**Finding (RQ3).** Austria's summer duck curve was essentially flat through 2022 and
then emerged abruptly in 2023–24: the midday belly deepened from ~390 MW below the
morning shoulder in 2022 to ~2,520 MW in 2024, the average evening ramp steepened from
~90 to ~460 MW/hour (785 MW in the steepest single hour), and the belly's low point
migrated two hours earlier (15h → 13h) while the summer peak pushed two hours later
(18h → 20h). The signature is deepest in summer but now appears in every season — even
winter's belly more than doubled, to ~1,200 MW — and its abrupt onset tracks the ~5×
solar build-out of RQ1, marking the duck as a solar-driven, post-2022 development rather
than a gradual trend.